# Exploratory data analysis

This notebook explicitly loads and explores each repository dataset. It intentionally avoids helper functions so every section can be edited and extended. Install with `python -m pip install pandas openpyxl geopandas matplotlib seaborn jupyter`.

Documentation rules applied: REFEPS `total` is an aggregate count; `99`, `999`, and `9999` are missing sentinels; BAHRA contains settlements; census-radio metadata documents EPSG:4326.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
ROOT = Path(".")
CSV_NA = ["", "NA", "N/A", "null", "NULL"]
print(ROOT.resolve())

## 1. Enfermería 2023

In [ ]:
enfermeria = pd.read_csv(ROOT / "enfermeria-2023.csv", encoding="utf-8", na_values=CSV_NA, dtype={"sexo":"string", "grupo_etareo":"string", "id_provincia_residencia":"Int64", "provincia_residencia":"string", "id_pais_nacimiento":"Int64", "pais_nacimiento":"string", "pais_origen":"string", "id_profesion_referencia":"Int64", "profesion_referencia":"string", "id_pais_formacion":"Int64", "pais_formacion":"string", "id_provincia_formacion":"Int64", "provincia_formacion":"string", "id_institucion_formadora":"Int64", "institucion_formadora":"string", "anio_titulo":"Int64", "total":"Int64", "anio_corte":"Int64"});
for column in ["id_pais_nacimiento", "id_pais_formacion", "id_provincia_residencia", "id_provincia_formacion", "anio_titulo", "id_institucion_formadora"]: enfermeria[column] = enfermeria[column].replace([99,999,9999], pd.NA).astype("Int64")
print(enfermeria.shape); display(enfermeria.head()); display(enfermeria.dtypes.to_frame("dtype"))

In [ ]:
enfermeria_missing = enfermeria.isna().sum().to_frame("missing"); enfermeria_missing["missing_pct"] = enfermeria_missing["missing"] / len(enfermeria) * 100; display(enfermeria_missing.sort_values("missing", ascending=False)); print("Duplicate rows:", enfermeria.duplicated().sum()); enfermeria_outliers=[]
for column in enfermeria.select_dtypes(include="number").columns:
 values=enfermeria[column].dropna().astype(float); q1,q3=values.quantile([.25,.75]); iqr=q3-q1; lower,upper=q1-1.5*iqr,q3+1.5*iqr; enfermeria_outliers.append([column,q1,q3,lower,upper,int(((values<lower)|(values>upper)).sum())])
display(pd.DataFrame(enfermeria_outliers, columns=["column","q1","q3","lower","upper","outlier_count"]))

In [ ]:
print("Professionals represented:", enfermeria["total"].sum()); display(enfermeria.groupby("sexo",dropna=False)["total"].sum().sort_values(ascending=False).to_frame("professionals")); display(enfermeria.groupby("grupo_etareo",dropna=False)["total"].sum().sort_index().to_frame("professionals")); display(enfermeria.groupby("provincia_residencia",dropna=False)["total"].sum().sort_values(ascending=False).head(15).to_frame("professionals")); display(enfermeria.groupby("pais_origen",dropna=False)["total"].sum().sort_values(ascending=False).to_frame("professionals")); fig,axes=plt.subplots(1,2,figsize=(15,5)); sns.histplot(data=enfermeria,x="total",bins=40,ax=axes[0]); axes[0].set_title("Enfermería: grouped counts"); enfermeria.groupby("provincia_residencia")["total"].sum().sort_values(ascending=False).head(15).sort_values().plot.barh(ax=axes[1]); axes[1].set_title("Enfermería: residence provinces"); plt.tight_layout(); plt.show()

## 2. Medicina 2023

In [ ]:
medicos = pd.read_csv(ROOT / "medicos-2023.csv", encoding="utf-8", na_values=CSV_NA, dtype={"sexo":"string", "grupo_etareo":"string", "id_provincia_residencia":"Int64", "provincia_residencia":"string", "id_pais_nacimiento":"Int64", "pais_nacimiento":"string", "pais_origen":"string", "id_profesion_referencia":"Int64", "profesion_referencia":"string", "id_pais_formacion":"Int64", "pais_formacion":"string", "id_provincia_formacion":"Int64", "provincia_formacion":"string", "id_institucion_formadora":"Int64", "institucion_formadora":"string", "anio_titulo":"Int64", "total":"Int64", "anio_corte":"Int64"});
for column in ["id_pais_nacimiento", "id_pais_formacion", "id_provincia_residencia", "id_provincia_formacion", "anio_titulo", "id_institucion_formadora"]: medicos[column] = medicos[column].replace([99,999,9999], pd.NA).astype("Int64")
print(medicos.shape); display(medicos.head()); display(medicos.dtypes.to_frame("dtype"))

In [ ]:
medicos_missing=medicos.isna().sum().to_frame("missing"); medicos_missing["missing_pct"]=medicos_missing["missing"]/len(medicos)*100; display(medicos_missing.sort_values("missing",ascending=False)); print("Duplicate rows:",medicos.duplicated().sum()); medicos_outliers=[]
for column in medicos.select_dtypes(include="number").columns:
 values=medicos[column].dropna().astype(float); q1,q3=values.quantile([.25,.75]); iqr=q3-q1; lower,upper=q1-1.5*iqr,q3+1.5*iqr; medicos_outliers.append([column,q1,q3,lower,upper,int(((values<lower)|(values>upper)).sum())])
display(pd.DataFrame(medicos_outliers,columns=["column","q1","q3","lower","upper","outlier_count"]))

In [ ]:
print("Professionals represented:",medicos["total"].sum()); display(medicos.groupby("sexo",dropna=False)["total"].sum().sort_values(ascending=False).to_frame("professionals")); display(medicos.groupby("grupo_etareo",dropna=False)["total"].sum().sort_index().to_frame("professionals")); display(medicos.groupby("provincia_residencia",dropna=False)["total"].sum().sort_values(ascending=False).head(15).to_frame("professionals")); display(medicos.groupby("pais_origen",dropna=False)["total"].sum().sort_values(ascending=False).to_frame("professionals")); fig,axes=plt.subplots(1,2,figsize=(15,5)); sns.histplot(data=medicos,x="total",bins=40,ax=axes[0]); axes[0].set_title("Medicine: grouped counts"); medicos.groupby("provincia_residencia")["total"].sum().sort_values(ascending=False).head(15).sort_values().plot.barh(ax=axes[1]); axes[1].set_title("Medicine: residence provinces"); plt.tight_layout(); plt.show()

## 3. REFES establishments XLSX

In [ ]:
refes = pd.read_excel(ROOT / "establecimientos-asistenciales-asentados-registro-federal-refes-20260114 (1).xlsx", sheet_name=0, na_values=CSV_NA, dtype="string"); refes["provincia_id"] = pd.to_numeric(refes["provincia_id"], errors="coerce").astype("Int64"); refes["departamento_id"] = pd.to_numeric(refes["departamento_id"], errors="coerce").astype("Int64"); refes["tipologia_id"] = pd.to_numeric(refes["tipologia_id"], errors="coerce").astype("Int64"); refes["longitud"] = pd.to_numeric(refes["longitud"], errors="coerce"); refes["latitud"] = pd.to_numeric(refes["latitud"], errors="coerce"); print(refes.shape); display(refes.head()); display(refes.dtypes.to_frame("dtype"))

In [ ]:
refes_missing=refes.isna().sum().to_frame("missing"); refes_missing["missing_pct"]=refes_missing["missing"]/len(refes)*100; display(refes_missing.sort_values("missing",ascending=False)); print("Duplicate rows:",refes.duplicated().sum()); print("Duplicate establishment IDs:",refes["establecimiento_id"].duplicated().sum()); refes_outliers=[]
for column in refes.select_dtypes(include="number").columns:
 values=refes[column].dropna(); q1,q3=values.quantile([.25,.75]); iqr=q3-q1; lower,upper=q1-1.5*iqr,q3+1.5*iqr; refes_outliers.append([column,q1,q3,lower,upper,int(((values<lower)|(values>upper)).sum())])
display(pd.DataFrame(refes_outliers,columns=["column","q1","q3","lower","upper","outlier_count"])); display(refes["origen_financiamiento"].value_counts(dropna=False).to_frame("establishments")); display(refes["provincia_nombre"].value_counts(dropna=False).head(15).to_frame("establishments")); display(refes["tipologia_sigla"].value_counts(dropna=False).head(20).to_frame("establishments"))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5)); refes["provincia_nombre"].value_counts().head(15).sort_values().plot.barh(ax=axes[0]); axes[0].set_title("REFES by province"); refes["tipologia_sigla"].value_counts().head(15).sort_values().plot.barh(ax=axes[1]); axes[1].set_title("REFES by typology"); plt.tight_layout(); plt.show(); refes_points=refes.dropna(subset=["longitud","latitud"]).query("longitud >= -75 and longitud <= -50 and latitud >= -56 and latitud <= -20"); print("Valid-looking coordinates used in map:",len(refes_points),"of",refes[["longitud","latitud"]].dropna().shape[0]); refes_map=gpd.GeoDataFrame(refes_points,geometry=gpd.points_from_xy(refes_points["longitud"],refes_points["latitud"]),crs="EPSG:4326"); refes_map.plot(figsize=(8,8),markersize=2,alpha=.4); plt.title("REFES establishments with valid-looking coordinates"); plt.show()

## 4. BAHRA GeoJSON

In [ ]:
bahra=gpd.read_file(ROOT / "base_total.geojson" / "base_total.geojson"); print("Shape:",bahra.shape,"CRS:",bahra.crs); display(bahra.head()); display(bahra.dtypes.to_frame("dtype")); bahra_missing=bahra.drop(columns="geometry").isna().sum().to_frame("missing"); bahra_missing["missing_pct"]=bahra_missing["missing"]/len(bahra)*100; display(bahra_missing.sort_values("missing",ascending=False)); print("Empty geometries:",bahra.geometry.is_empty.sum(),"Invalid geometries:",(~bahra.geometry.is_valid).sum()); display(bahra["tipo"].value_counts(dropna=False).to_frame("settlements")); display(bahra.groupby(["nom_pcia","tipo"],dropna=False).size().unstack(fill_value=0).head(15))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,7)); bahra.plot(ax=axes[0],column="tipo",markersize=2,alpha=.5,legend=True); axes[0].set_title("BAHRA settlements by type"); axes[0].set_axis_off(); bahra["nom_pcia"].value_counts().head(15).sort_values().plot.barh(ax=axes[1]); axes[1].set_title("BAHRA settlements by province"); plt.tight_layout(); plt.show()

## 5. Census radios Shapefile

In [ ]:
radios=gpd.read_file(ROOT / "radios_censales2" / "radios_censales2.shp",encoding="cp1252"); print("Shape:",radios.shape); print("CRS reported by file:",radios.crs); print("CRS documented in metadatos_radios_censales.pdf: EPSG:4326"); display(radios.head()); display(radios.dtypes.to_frame("dtype")); radios_missing=radios.drop(columns="geometry").isna().sum().to_frame("missing"); radios_missing["missing_pct"]=radios_missing["missing"]/len(radios)*100; display(radios_missing.sort_values("missing",ascending=False)); print("Duplicate radio IDs:",radios["id"].duplicated().sum()); print("Empty geometries:",radios.geometry.is_empty.sum(),"Invalid geometries:",(~radios.geometry.is_valid).sum()); display(radios["tro"].value_counts(dropna=False).to_frame("radios")); display(radios.groupby(["jur","tro"],dropna=False).size().unstack(fill_value=0).head(15))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,7)); radios.plot(ax=axes[0],column="tro",legend=True,linewidth=.05); axes[0].set_title("Census radios by type"); axes[0].set_axis_off(); radios["jur"].value_counts().head(15).sort_values().plot.barh(ax=axes[1]); axes[1].set_title("Census radios by jurisdiction"); plt.tight_layout(); plt.show()

## Notes for continued work

The objects `enfermeria`, `medicos`, `refes`, `refes_map`, `bahra`, and `radios` remain available. Use `sum(total)` for professional counts. Resolve the census-radio CRS discrepancy before spatial joins; the notebook reports the CRS rather than overwriting it.